<a href="https://colab.research.google.com/github/marcplanas11-alt/insurance-claims-triage-ai/blob/main/Copia_de_claims_langgraph_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claims AI Workflow (LangGraph + Claude)

## 1. Setup
## 2. State Definition
## 3. Claude Decision Agent
## 4. Workflow Nodes
## 5. Graph Construction
## 6. Test Suite
## 7. Results

## Mental model

State = evolving claim file.

Nodes = workflow steps that update the claim file:
- decision_node: Claude makes initial claim decision
- validation_node: Python checks whether output is valid
- routing_node: Python decides operational route
- human_review_node: sends uncertain claim to human adjuster

Edges = arrows between workflow steps.

Conditional edges = routing rules based on state.

Claude thinks.
Python controls.
LangGraph orchestrates.
State stores the audit trail.

LangGraph = tool that controls node order and conditional flow

# Claims Agentic Workflow with LangGraph

Goal:
Convert my existing claims workflow into a state-machine-based agentic system.

Insurance use case:
Claims triage with auditability, validation, routing, and human-in-the-loop escalation.

Architecture:
State → Decision Node → Validation Node → Routing Node → Human Review if needed

TypedDict = dictionary with expected fields
Optional = field can be empty / None
List = list of items
Dict = dictionary
Any = flexible value

## My understanding of ClaimState

ClaimState is the digital claim file.

At the beginning it contains:
- claim_id
- claim_text

Each node adds information to the same claim file:
- decision_node adds decision, confidence, and reason
- validation_node adds validation result and errors
- routing_node adds routing and final status

This is useful because the final state shows what happened, why it happened, and whether human review was needed.

ClaimState = evolving digital claim file

## LangGraph Claims Workflow Summary

I built a state-machine claims workflow using LangGraph.

The workflow includes:
- decision_node: makes an initial claims decision
- validation_node: checks decision quality and required fields
- routing_node: applies deterministic claims routing

Governance controls:
- JSON-like structured state
- confidence threshold
- validation layer
- fallback to manual review
- audit log for every node

Insurance relevance:
This mirrors a controlled MGA claims triage process where AI supports decisions but Python routing and human-in-the-loop controls govern the final outcome.

In [3]:
!pip install -q langgraph langchain langchain-core anthropic

In [4]:
import os
from getpass import getpass
from anthropic import Anthropic

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

client = Anthropic()

Enter your Anthropic API key: ··········


In [42]:
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, START, END
import json

class ClaimState(TypedDict):
    claim_id: str
    claim_text: str

    decision: Optional[str]
    confidence: Optional[float]
    decision_reason: Optional[str]

    is_valid: Optional[bool]
    validation_errors: Optional[List[str]]

    routing: Optional[str]
    final_status: Optional[str]

    human_review_required: bool
    human_review_reason: Optional[str]
    human_reviewer: Optional[str]
    human_review_status: Optional[str]

    audit_log: List[Dict[str, Any]]

# --- Helper functions ---
def safe_json_parse(raw_text: str) -> dict:
    cleaned = raw_text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    elif cleaned.startswith("```"):
        cleaned = cleaned.replace("```", "").strip()
    return json.loads(cleaned)

def decision_agent_claude(claim_text: str) -> dict:
    prompt = f"""
You are an insurance claims triage assistant.

Analyze the claim text and return ONLY valid JSON.

Allowed decisions:
- APPROVE
- REJECT
- ESCALATE

Rules:
- APPROVE: Low-risk, common claims with clear damage descriptions (e.g., minor vehicle damage, water damage, small incidents), even if full policy details are not provided.
- REJECT: Only if there is a clear indication that the claim is invalid.
- ESCALATE: When the claim is unclear, high-value or complex, has potential coverage ambiguity, or potential fraud signals.
- Do not invent facts.
- Do not assume coverage beyond reasonable common cases.

Claim text:
{claim_text}

Required JSON format:
{{
  "decision": "APPROVE | REJECT | ESCALATE",
  "confidence": 0.0,
  "reason": "short explanation"
}}
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    raw_text = response.content[0].text
    return safe_json_parse(raw_text)

# --- Node definitions ---
def decision_node(state: ClaimState) -> ClaimState:
    claim_text = state["claim_text"]
    try:
        result = decision_agent_claude(claim_text)
        decision = result.get("decision", "ESCALATE")
        confidence = result.get("confidence", 0.0)
        reason = result.get("reason", "No reason provided")
    except Exception as e:
        decision = "ESCALATE"
        confidence = 0.0
        reason = f"Claude decision failed: {str(e)}"

    state["decision"] = decision
    state["confidence"] = confidence
    state["decision_reason"] = reason

    # UPDATED: If decision is REJECT or ESCALATE, or low confidence, escalate to human
    escalate_criteria = (decision == "ESCALATE" or decision == "REJECT" or confidence < 0.7)
    state["human_review_required"] = escalate_criteria

    state["audit_log"].append({
        "step": "decision_node",
        "agent": "claude",
        "decision": decision,
        "confidence": confidence,
        "reason": reason
    })
    return state

def validation_node(state: ClaimState) -> ClaimState:
    errors = []
    if state["decision"] not in ["APPROVE", "REJECT", "ESCALATE"]:
        errors.append("Invalid decision value")
    if state["confidence"] is None or not (0 <= state["confidence"] <= 1):
        errors.append("Invalid or missing confidence score")

    state["is_valid"] = len(errors) == 0
    state["validation_errors"] = errors
    if not state["is_valid"]:
        state["human_review_required"] = True

    state["audit_log"].append({"step": "validation_node", "is_valid": state["is_valid"], "errors": errors})
    return state

def routing_node(state: ClaimState) -> ClaimState:
    if state["decision"] == "APPROVE":
        routing = "AUTO_PAY"
        final_status = "Approved for automatic payment"
    else:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated"

    state["routing"] = routing
    state["final_status"] = final_status
    state["audit_log"].append({"step": "routing_node", "routing": routing, "final_status": final_status})
    return state

def human_review_node(state: ClaimState) -> ClaimState:

    review_reason = state.get("decision_reason", "Manual review required")

    state["human_review_reason"] = review_reason
    state["human_reviewer"] = "claims_adjuster"
    state["human_review_status"] = "PENDING"
    state["routing"] = "MANUAL_REVIEW"
    state["final_status"] = "Pending human claims review"

    state["audit_log"].append({
        "step": "human_review_node",
        "review_required": True,
        "review_reason": review_reason,
        "assigned_to": "claims_adjuster",
        "status": "PENDING"
    })

    return state

# --- Graph Construction ---
workflow = StateGraph(ClaimState)
workflow.add_node("decision", decision_node)
workflow.add_node("validation", validation_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("routing", routing_node)

workflow.add_edge(START, "decision")
workflow.add_edge("decision", "validation")
workflow.add_conditional_edges(
    "validation",
    decide_review_or_routing,
    {"human_review": "human_review", "routing": "routing"}
)
workflow.add_edge("human_review", END)
workflow.add_edge("routing", END)

claims_graph = workflow.compile()

In [43]:
initial_state = {
    "claim_id": "C4",
    "claim_text": "Water damage in kitchen after pipe burst",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

# Fix: Invoke the claims_graph to get the final_state, as done in other cells.
final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C4', 'claim_text': 'Water damage in kitchen after pipe burst', 'decision': 'APPROVE', 'confidence': 0.85, 'decision_reason': 'Common, low-risk claim with clear damage description (water damage from pipe burst). Standard homeowners insurance typically covers sudden, accidental water damage from burst pipes. No fraud signals or coverage ambiguity present.', 'is_valid': True, 'validation_errors': [], 'routing': 'AUTO_PAY', 'final_status': 'Approved for automatic payment', 'human_review_required': False, 'human_review_reason': None, 'human_reviewer': None, 'human_review_status': None, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'APPROVE', 'confidence': 0.85, 'reason': 'Common, low-risk claim with clear damage description (water damage from pipe burst). Standard homeowners insurance typically covers sudden, accidental water damage from burst pipes. No fraud signals or coverage ambiguity present.'}, {'step': 'validation_node', 'is_valid': True, 'error

In [44]:
initial_state = {
    "claim_id": "C5",
    "claim_text": "Customer reports unusual situation with unclear details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

  "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C5', 'claim_text': 'Customer reports unusual situation with unclear details', 'decision': 'ESCALATE', 'confidence': 0.95, 'decision_reason': 'Claim lacks specific details about the incident, damage type, or circumstances. Insufficient information to assess validity or coverage applicability.', 'is_valid': True, 'validation_errors': [], 'routing': 'MANUAL_REVIEW', 'final_status': 'Pending human claims review', 'human_review_required': True, 'human_review_reason': 'Claim lacks specific details about the incident, damage type, or circumstances. Insufficient information to assess validity or coverage applicability.', 'human_reviewer': 'claims_adjuster', 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.95, 'reason': 'Claim lacks specific details about the incident, damage type, or circumstances. Insufficient information to assess validity or coverage applicability.'}, {'step': 'validation_node'

In [45]:
initial_state = {
    "claim_id": "C6",
    "claim_text": "Minor scratch on insured vehicle reported with full details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,
"human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C6', 'claim_text': 'Minor scratch on insured vehicle reported with full details', 'decision': 'APPROVE', 'confidence': 0.85, 'decision_reason': 'Low-risk claim with clear damage description (minor scratch). Common vehicle damage claim with full details provided.', 'is_valid': True, 'validation_errors': [], 'routing': 'AUTO_PAY', 'final_status': 'Approved for automatic payment', 'human_review_required': False, 'human_review_reason': None, 'human_reviewer': None, 'human_review_status': None, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'APPROVE', 'confidence': 0.85, 'reason': 'Low-risk claim with clear damage description (minor scratch). Common vehicle damage claim with full details provided.'}, {'step': 'validation_node', 'is_valid': True, 'errors': []}, {'step': 'routing_node', 'routing': 'AUTO_PAY', 'final_status': 'Approved for automatic payment'}]}


In [46]:
initial_state = {
    "claim_id": "C5",
    "claim_text": "Customer reports unusual situation with unclear details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C5', 'claim_text': 'Customer reports unusual situation with unclear details', 'decision': 'ESCALATE', 'confidence': 0.95, 'decision_reason': 'Claim lacks specific details about the incident, damage type, and circumstances. Insufficient information to assess validity or coverage applicability.', 'is_valid': True, 'validation_errors': [], 'routing': 'MANUAL_REVIEW', 'final_status': 'Pending human claims review', 'human_review_required': True, 'human_review_reason': 'Claim lacks specific details about the incident, damage type, and circumstances. Insufficient information to assess validity or coverage applicability.', 'human_reviewer': 'claims_adjuster', 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.95, 'reason': 'Claim lacks specific details about the incident, damage type, and circumstances. Insufficient information to assess validity or coverage applicability.'}, {'step': 'validation_no

In [47]:
initial_state = {
    "claim_id": "C6",
    "claim_text": "Minor scratch on insured vehicle reported with full details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C6', 'claim_text': 'Minor scratch on insured vehicle reported with full details', 'decision': 'APPROVE', 'confidence': 0.85, 'decision_reason': 'Low-risk claim for minor vehicle damage with full details provided. Consistent with common, straightforward claims suitable for approval.', 'is_valid': True, 'validation_errors': [], 'routing': 'AUTO_PAY', 'final_status': 'Approved for automatic payment', 'human_review_required': False, 'human_review_reason': None, 'human_reviewer': None, 'human_review_status': None, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'APPROVE', 'confidence': 0.85, 'reason': 'Low-risk claim for minor vehicle damage with full details provided. Consistent with common, straightforward claims suitable for approval.'}, {'step': 'validation_node', 'is_valid': True, 'errors': []}, {'step': 'routing_node', 'routing': 'AUTO_PAY', 'final_status': 'Approved for automatic payment'}]}


In [48]:
initial_state = {
    "claim_id": "C6",
    "claim_text": "Broken windshield due to insured's intentional behaviour",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

{'claim_id': 'C6', 'claim_text': "Broken windshield due to insured's intentional behaviour", 'decision': 'REJECT', 'confidence': 0.95, 'decision_reason': "Claim explicitly states damage was caused by insured's intentional behaviour. Intentional damage is a standard exclusion in insurance policies and is not a valid claim.", 'is_valid': True, 'validation_errors': [], 'routing': 'MANUAL_REVIEW', 'final_status': 'Pending human claims review', 'human_review_required': True, 'human_review_reason': "Claim explicitly states damage was caused by insured's intentional behaviour. Intentional damage is a standard exclusion in insurance policies and is not a valid claim.", 'human_reviewer': 'claims_adjuster', 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'REJECT', 'confidence': 0.95, 'reason': "Claim explicitly states damage was caused by insured's intentional behaviour. Intentional damage is a standard exclusion in insurance policies and 

In [49]:
initial_state_reject = {
    "claim_id": "C7_REJECT_TEST",
    "claim_text": "I broke my own television with a hammer because I wanted a newer model. I am reporting this as a claim.",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

# This should trigger REJECT + high confidence -> human_review_required = True
final_state_reject = claims_graph.invoke(initial_state_reject)

print(f"Decision: {final_state_reject['decision']}")
print(f"Confidence: {final_state_reject['confidence']}")
print(f"Human Review Required: {final_state_reject['human_review_required']}")
print(f"Routing: {final_state_reject['routing']}")
print(f"Final Status: {final_state_reject['final_status']}")

Decision: REJECT
Confidence: 0.95
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review


In [50]:
initial_state_reject_test = {
    "claim_id": "C8_REJECT_TEST",
    "claim_text": "I accidentally dropped my phone into the ocean while on vacation, and I want to claim for a new house.",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

# This should trigger REJECT -> human_review_required = True due to the updated logic
final_state_reject_test = claims_graph.invoke(initial_state_reject_test)

print(f"Decision: {final_state_reject_test['decision']}")
print(f"Human Review Required: {final_state_reject_test['human_review_required']}")
print(f"Routing: {final_state_reject_test['routing']}")
print(f"Final Status: {final_state_reject_test['final_status']}")

Decision: REJECT
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review


In [51]:
initial_state_test_flow = {
    "claim_id": "TEST_SEQUENCE_001",
    "claim_text": "I intentionally broke my window to test the system.",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "human_review_reason": None,
    "human_reviewer": None,
    "human_review_status": None,

    "audit_log": []
}

# Invoke the graph
final_state_test = claims_graph.invoke(initial_state_test_flow)

print(f"Final Status: {final_state_test['final_status']}")
print("\nExecution Sequence (Audit Log):")
for entry in final_state_test['audit_log']:
    print(f"- {entry['step']}")

Final Status: Pending human claims review

Execution Sequence (Audit Log):
- decision_node
- validation_node
- human_review_node


In [52]:
def run_claim_test(claim_id: str, claim_text: str):

    initial_state = {
        "claim_id": claim_id,
        "claim_text": claim_text,

        "decision": None,
        "confidence": None,
        "decision_reason": None,

        "is_valid": None,
        "validation_errors": None,

        "routing": None,
        "final_status": None,

        "human_review_required": False,
        "human_review_reason": None,
        "human_reviewer": None,
        "human_review_status": None,

        "audit_log": []
    }

    final_state = claims_graph.invoke(initial_state)

    print("=" * 80)
    print(f"Claim ID: {final_state['claim_id']}")
    print(f"Claim Text: {final_state['claim_text']}")
    print(f"Decision: {final_state['decision']}")
    print(f"Confidence: {final_state['confidence']}")
    print(f"Human Review Required: {final_state['human_review_required']}")
    print(f"Routing: {final_state['routing']}")
    print(f"Final Status: {final_state['final_status']}")
    print("\nAudit Log:")

    for entry in final_state["audit_log"]:
        print(f"- {entry['step']}")

    return final_state

In [53]:
auto_pay_result = run_claim_test(
    "TEST_AUTO_PAY_001",
    "Minor scratch on insured vehicle reported with full details"
)

Claim ID: TEST_AUTO_PAY_001
Claim Text: Minor scratch on insured vehicle reported with full details
Decision: APPROVE
Confidence: 0.85
Human Review Required: False
Routing: AUTO_PAY
Final Status: Approved for automatic payment

Audit Log:
- decision_node
- validation_node
- routing_node


In [55]:
claims_graph = workflow.compile()

In [56]:
manual_review_result = run_claim_test(
    "TEST_MANUAL_REVIEW_001",
    "Customer reports unusual situation with unclear details"
)

Claim ID: TEST_MANUAL_REVIEW_001
Claim Text: Customer reports unusual situation with unclear details
Decision: ESCALATE
Confidence: 0.95
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review

Audit Log:
- decision_node
- validation_node
- human_review_node


In [57]:
reject_review_result = run_claim_test(
    "TEST_REJECT_REVIEW_001",
    "I intentionally broke my window to test the system."
)

Claim ID: TEST_REJECT_REVIEW_001
Claim Text: I intentionally broke my window to test the system.
Decision: REJECT
Confidence: 0.95
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review

Audit Log:
- decision_node
- validation_node
- human_review_node


In [59]:
# Se corrigieron los argumentos para pasar strings válidos o variables existentes
auto_pay_result = run_claim_test("RE-TEST-AUTO", "Minor scratch on vehicle")
manual_review_result = run_claim_test("RE-TEST-MANUAL", "Unclear situation details")
reject_review_result = run_claim_test("RE-TEST-REJECT", "Intentional damage reported")

Claim ID: RE-TEST-AUTO
Claim Text: Minor scratch on vehicle
Decision: APPROVE
Confidence: 0.85
Human Review Required: False
Routing: AUTO_PAY
Final Status: Approved for automatic payment

Audit Log:
- decision_node
- validation_node
- routing_node
Claim ID: RE-TEST-MANUAL
Claim Text: Unclear situation details
Decision: ESCALATE
Confidence: 0.95
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review

Audit Log:
- decision_node
- validation_node
- human_review_node
Claim ID: RE-TEST-REJECT
Claim Text: Intentional damage reported
Decision: REJECT
Confidence: 0.95
Human Review Required: True
Routing: MANUAL_REVIEW
Final Status: Pending human claims review

Audit Log:
- decision_node
- validation_node
- human_review_node
